# S&P 500 Next-Day Direction: Walk-Forward Research Baseline

## Project snapshot

| | |
|---|---|
| **Goal** | Test whether lagged price and trend features contain signal for the next S&P 500 closing direction. |
| **Data** | Daily OHLCV history for `^GSPC` from Yahoo Finance, restricted to 1990 onward. |
| **Approach** | Random-forest baseline, expanding-window backtest, multi-horizon ratios/trends, and a higher positive-class threshold. |
| **Evaluation** | Intended out-of-sample precision and prediction coverage, compared with the observed up-day prevalence. |
| **Status** | Research workflow implemented but not executed in this repository; no predictive or trading performance is claimed. |

> **Research scope:** This is an educational classification experiment, not financial advice or evidence of a profitable strategy. Run top-to-bottom with network access on the first pass; data are then cached under the repository's shared `datasets/` directory.


## 1. Imports & data loading
Load dependencies and fetch historical data (with local CSV caching).


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import yfinance as yf

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "portfolio_utils").is_dir():
        repo_root = candidate
        break
else:
    raise FileNotFoundError("Could not locate the repository root.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from portfolio_utils import DATASETS_DIR

In [ ]:
cache_path = DATASETS_DIR / "sp500.csv"

if cache_path.exists():
    sp500 = pd.read_csv(cache_path, index_col=0)
else:
    sp500 = yf.Ticker("^GSPC")
    sp500 = sp500.history(period="max")
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    sp500.to_csv(cache_path)

In [ ]:
sp500.index = pd.to_datetime(sp500.index)

In [ ]:
sp500

## 2. Quick visual check
Plot the S&P 500 close price to sanity‑check the dataset.


In [ ]:
sp500.plot.line(y="Close", use_index=True)

In [ ]:
del sp500["Dividends"]
del sp500["Stock Splits"]

## 3. Define the prediction target
Create:
- `Tomorrow`: next day’s close (shifted by -1)
- `Target`: 1 if tomorrow’s close is greater than today’s close, else 0

The final row has no observed next-day close, so it is removed before creating the target.


In [ ]:
sp500["Tomorrow"] = sp500["Close"].shift(-1)
sp500 = sp500.dropna(subset=["Tomorrow"]).copy()

In [ ]:
sp500["Target"] = (sp500["Tomorrow"] > sp500["Close"]).astype(int)

In [ ]:
sp500 = sp500.loc["1990-01-01":].copy()

In [ ]:
sp500

## 4. Baseline model
Train a baseline Random Forest on a small set of raw OHLCV features and evaluate on a holdout split.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, min_samples_split=100, random_state=1)

train = sp500.iloc[:-100]
test = sp500.iloc[-100:]

predictors = ["Close", "Volume", "Open", "High", "Low"]
model.fit(train[predictors], train["Target"])

In [ ]:
from sklearn.metrics import precision_score

preds = model.predict(test[predictors])
preds = pd.Series(preds, index=test.index)
precision_score(test["Target"], preds)

In [ ]:
# label the predictions with the actual price movement
combined = pd.concat([test["Target"], preds], axis=1)
combined.columns = ["Target", "Predictions"]
combined.plot()

In [ ]:
combined

## 5. Helper: prediction function
Wrap training + inference into a reusable function that returns a tidy `Target` vs `Predictions` dataframe.


In [ ]:
# Create a function to make predictions and label them with the actual price movement
def predict(train, test, predictors, model):
    model.fit(train[predictors], train["Target"])
    preds = model.predict(test[predictors])
    preds = pd.Series(preds, index=test.index, name="Predictions")
    combined = pd.concat([test["Target"], preds], axis=1)
    combined.columns = ["Target", "Predictions"]
    return combined

## 6. Rolling backtest
Evaluate the model with an expanding-window backtest so each prediction block is trained only on earlier observations.


In [ ]:
# Backtest the model using a rolling window approach
def backtest(data, model, predictors, start=2500, step=250):
    all_predictions = []

    for i in range(start, data.shape[0], step):
        train = data.iloc[0:i].copy()
        test = data.iloc[i:(i+step)].copy()
        predictions = predict(train, test, predictors, model)
        all_predictions.append(predictions)
    
    return pd.concat(all_predictions)

In [ ]:
predictions = backtest(sp500, model, predictors)

In [ ]:
predictions

In [ ]:
predictions["Predictions"].value_counts()

In [ ]:
precision_score(predictions["Target"], predictions["Predictions"])

In [ ]:
predictions["Target"].value_counts() / predictions.shape[0]

## 7. Feature engineering across multiple horizons
Create features that capture:
- **Mean reversion**: today’s close relative to a rolling mean (`Close_Ratio_*`)
- **Momentum/trend**: recent up‑day counts (`Trend_*`) using `Target` shifted by 1 day to avoid leakage


In [ ]:
horizons = [2,5,60,250,1000]
new_predictors = []

for horizon in horizons:
    rolling_averages = sp500.rolling(horizon).mean()
    
    ratio_column = f"Close_Ratio_{horizon}"
    sp500[ratio_column] = sp500["Close"] / rolling_averages["Close"]
    
    trend_column = f"Trend_{horizon}"
    sp500[trend_column] = sp500.shift(1).rolling(horizon).sum()["Target"]
    
    new_predictors+= [ratio_column, trend_column]

## 8. Final dataset cleanup
Drop early rows where rolling features are undefined. The unlabeled final row was already removed before target construction.


In [ ]:
sp500 = sp500.dropna()

In [ ]:
sp500

## 9. Updated model & probability thresholding
Switch to probability predictions (`predict_proba`) and apply a threshold (0.6) to trade off precision vs coverage.


In [ ]:
model = RandomForestClassifier(n_estimators=200, min_samples_split=50, random_state=1)

In [ ]:
def predict(train, test, predictors, model):
    model.fit(train[predictors], train["Target"])
    preds = model.predict_proba(test[predictors])[:,1]
    preds[preds >=.6] = 1
    preds[preds <.6] = 0
    preds = pd.Series(preds, index=test.index, name="Predictions")
    combined = pd.concat([test["Target"], preds], axis=1)
    return combined

In [ ]:
predictions = backtest(sp500, model, new_predictors)

In [ ]:
predictions["Predictions"].value_counts()

In [ ]:
precision_score(predictions["Target"], predictions["Predictions"])

In [ ]:
predictions["Target"].value_counts() / predictions.shape[0]

In [ ]:
predictions

---

## Results and takeaways

The notebook implements a chronological research design and compares model precision with the natural up-day rate rather than relying on accuracy alone. The multi-horizon features use only information available before the prediction day. Because the checked-in notebook is unexecuted, the precision and coverage cells represent an evaluation plan, not observed evidence of signal.

## Limitations and next steps

- The `0.60` probability threshold is exploratory and is not tuned or calibrated on a separate validation period.
- Precision alone omits recall, calibration, turnover, drawdown, transaction costs, and the economic size of predicted moves.
- Yahoo Finance history and index composition can be revised; cache the exact extract and record its retrieval date for reproducible comparisons.
- Add a fixed final test window, threshold selection inside each training fold, naive directional baselines, and a cost-aware return simulation before drawing practical conclusions.
